# EEG Emotion Recognition - Notebook 03: Model A (CNN Baseline)

## Model A Architecture: Convolutional Neural Network
- **Purpose:** Serves as the spatial/local temporal feature baseline for 32-channel EEG multi-output regression.
- **Input:** `(batch_size, 256 timesteps, 32 channels)`
- **Key Layers:**
  - `Conv1D(32, kernel_size=5)` + BatchNorm + ReLU + MaxPool1D(2) + Dropout(0.2)
  - `Conv1D(64, kernel_size=3)` + BatchNorm + ReLU + MaxPool1D(2) + Dropout(0.2)
  - `Conv1D(128, kernel_size=3)` + BatchNorm + ReLU + GlobalAveragePooling1D
  - `Dense(128)` + Dropout(0.3) -> `Dense(64)` -> `Dense(4, linear)` predicting **[Valence, Arousal, Dominance, Liking]**.
- **Loss Function:** Huber Loss (Smooth L1) with Adam optimizer.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/DEAP/processed'
    CHECKPOINT_DIR = '/content/drive/MyDrive/DEAP/checkpoints/cnn'
else:
    DATA_DIR = '../data/processed' if os.path.exists('../data/processed') else './data/processed'
    CHECKPOINT_DIR = '../models/checkpoints/cnn' if os.path.exists('../models') else './models/checkpoints/cnn'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import CONFIG
print("Setup completed successfully.")

In [ ]:
# Load Preprocessed Data Splits
train_data = np.load(os.path.join(DATA_DIR, 'train_data.npz'))
val_data = np.load(os.path.join(DATA_DIR, 'val_data.npz'))
test_data = np.load(os.path.join(DATA_DIR, 'test_data.npz'))

X_train, y_train = train_data['X'], train_data['y']
X_val, y_val = val_data['X'], val_data['y']
X_test, y_test = test_data['X'], test_data['y']

print(f"Training set:   X={X_train.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val.shape}, y={y_val.shape}")
print(f"Testing set:    X={X_test.shape}, y={y_test.shape}")

In [ ]:
# Build and Compile Keras CNN Model (TensorFlow / Colab T4)
try:
    import tensorflow as tf
    from src.models.cnn import build_keras_cnn
    
    model_cnn = build_keras_cnn(input_shape=(256, 32), num_outputs=4, dropout_rate=0.3)
    model_cnn.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
    )
    model_cnn.summary()
except ImportError:
    print("TensorFlow not installed in current environment. In Google Colab, TensorFlow runs with T4 GPU.")

In [ ]:
# Train Model A (CNN)
if 'model_cnn' in locals():
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
        tf.keras.callbacks.ModelCheckpoint(os.path.join(CHECKPOINT_DIR, 'best_cnn_model.keras'), save_best_only=True)
    ]
    
    history = model_cnn.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=CONFIG.training.batch_size,
        epochs=CONFIG.training.epochs,
        callbacks=callbacks,
        verbose=1
    )